In [0]:
%pip install xgboost

In [0]:
import pandas as pd
import numpy as np
import pickle
import boto3
import json
from io import BytesIO
from datetime import datetime, timezone

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
from xgboost import XGBRegressor

# =============================================================
# 1. CARGAR DATOS GOLD
# =============================================================
# Credenciales: Databricks secrets (producción) → archivo local (desarrollo)
try:
    config = {
        "aws_access_key": dbutils.secrets.get(scope="aws", key="access_key"),
        "aws_secret_key": dbutils.secrets.get(scope="aws", key="secret_key"),
        "bucket_name": "bronce-scrap-date",
    }
except Exception:
    with open("aws_secrets.json", "r") as f:
        config = json.load(f)

BUCKET = config["bucket_name"]
S3_OPTS = {
    "fs.s3a.access.key": config["aws_access_key"],
    "fs.s3a.secret.key": config["aws_secret_key"],
    "fs.s3a.endpoint": "s3.amazonaws.com",
}

ruta_gold = f"s3a://{BUCKET}/gold/app_inmuebles/"
reader = spark.read.format("parquet")
for k, v in S3_OPTS.items():
    reader = reader.option(k, v)
df_spark = reader.load(ruta_gold)

print(f"📊 Columnas Gold: {df_spark.columns}")

# =============================================================
# 2. PREPARAR FEATURES
# =============================================================
# Seleccionar todas las columnas útiles disponibles
cols_target = ["precio_num"]
cols_numeric = ["area_m2", "habitaciones", "banos", "garajes"]
cols_categorical = ["tipo_inmueble", "estado_inmueble", "fuente"]
cols_text = ["ubicacion_raw", "titulo"]

# Solo usar columnas que existan en Gold
available = set(df_spark.columns)
cols_numeric = [c for c in cols_numeric if c in available]
cols_categorical = [c for c in cols_categorical if c in available]
cols_text = [c for c in cols_text if c in available]

all_cols = cols_target + cols_numeric + cols_categorical + cols_text
print(f"📋 Features: numeric={cols_numeric}, cat={cols_categorical}, text={cols_text}")

df = df_spark.select(*all_cols).toPandas()
print(f"   Total registros: {len(df)}")

# Construir feature de texto
text_parts = []
for c in cols_text:
    text_parts.append(df[c].fillna(""))
df["texto_completo"] = text_parts[0]
for part in text_parts[1:]:
    df["texto_completo"] = df["texto_completo"] + " " + part

# Limpiar categóricos
for c in cols_categorical:
    df[c] = df[c].fillna("desconocido")

# Limpiar numéricos
for c in cols_numeric:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# =============================================================
# 3. FILTRAR OUTLIERS
# =============================================================
df_clean = df[
    (df["precio_num"] > df["precio_num"].quantile(0.05)) &
    (df["precio_num"] < df["precio_num"].quantile(0.95)) &
    (df["area_m2"] > 15) & (df["area_m2"] < 1000)
].copy()

print(f"🧹 Training set: {len(df_clean)} registros (de {len(df)})")

# =============================================================
# 4. PIPELINE ML
# =============================================================
# Seleccionar features numéricas con suficientes datos (>30% no-nulos)
num_features_final = []
for c in cols_numeric:
    pct_valid = df_clean[c].notna().mean()
    if pct_valid > 0.3:
        num_features_final.append(c)
        print(f"  ✅ {c}: {pct_valid:.0%} válidos")
    else:
        print(f"  ⏩ {c}: {pct_valid:.0%} válidos (descartado, <30%)")

# Seleccionar features categóricas con variabilidad
cat_features_final = []
for c in cols_categorical:
    n_unique = df_clean[c].nunique()
    if n_unique > 1:
        cat_features_final.append(c)
        print(f"  ✅ {c}: {n_unique} categorías")

# Construir transformadores
transformers = [
    ("txt", TfidfVectorizer(
        max_features=300,
        stop_words=["en", "de", "la", "el", "y", "con", "se", "del", "por", "los", "las", "un", "una"]
    ), "texto_completo"),
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ]), num_features_final),
]

if cat_features_final:
    transformers.append(
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="desconocido")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), cat_features_final)
    )

preprocessor = ColumnTransformer(transformers=transformers)

# Features finales para el modelo
feature_cols = num_features_final + cat_features_final + ["texto_completo"]
X = df_clean[feature_cols]
y = df_clean["precio_num"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# XGBoost con log-transform en target (precios tienen distribución log-normal)
xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", TransformedTargetRegressor(
        regressor=xgb, func=np.log1p, inverse_func=np.expm1
    )),
])

# =============================================================
# 5. ENTRENAR + EVALUAR
# =============================================================
print("\n🚀 Entrenando XGBoost con features enriquecidas...")
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
r2 = model.score(X_test, y_test)
mae = mean_absolute_error(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred) * 100

print(f"\n📊 RESULTADOS:")
print(f"   🌟 R² Score:  {r2:.4f}")
print(f"   💰 MAE:       ${mae:,.0f} COP")
print(f"   📏 MAPE:      {mape:.1f}%")

# Cross-validation (5-fold)
print("\n⏳ Cross-validation (5-fold)...")
cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring="r2")
print(f"   CV R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# =============================================================
# 6. GUARDAR MODELO EN S3 (Champion / Challenger)
# =============================================================
if r2 > 0.0:
    s3 = boto3.client(
        "s3",
        aws_access_key_id=config["aws_access_key"],
        aws_secret_access_key=config["aws_secret_key"],
    )

    # 1. Nombre versionado — nunca sobreescribe
    ts = datetime.now(tz=timezone.utc).strftime("%Y%m%d_%H%M%S")
    file_key = f"models/modelo_xgboost_{ts}.pkl"

    buffer = BytesIO()
    pickle.dump(model, buffer)
    buffer.seek(0)
    s3.upload_fileobj(buffer, BUCKET, file_key)
    print(f"\n💾 Modelo guardado: s3://{BUCKET}/{file_key}")

    # 2. Leer manifest anterior para comparar con campeón
    manifest_key = "models/manifest.json"
    try:
        prev = json.loads(
            s3.get_object(Bucket=BUCKET, Key=manifest_key)["Body"].read()
        )
        prev_mape = prev.get("metrics", {}).get("mape", float("inf"))
    except s3.exceptions.NoSuchKey:
        prev_mape = float("inf")
        prev = {}

    # 3. Solo actualizar el manifest si este modelo es mejor (o es el primero)
    MAPE_THRESHOLD = 0.5  # pp mínimos de mejora para desplazar al campeón

    if (prev_mape - mape) >= MAPE_THRESHOLD or prev_mape == float("inf"):
        manifest = {
            "champion_model_key": file_key,
            "champion_model_uri": f"s3://{BUCKET}/{file_key}",
            "metrics": {
                "r2": r2,
                "mae": mae,
                "mape": mape,
                "cv_r2_mean": float(cv_scores.mean()),
                "cv_r2_std": float(cv_scores.std()),
                "train_size": len(X_train),
                "test_size": len(X_test),
                "feature_cols": feature_cols,
            },
            "deployed_at": datetime.now(tz=timezone.utc).isoformat(),
            "previous_champion": prev.get("champion_model_key"),
            "previous_mape": prev_mape if prev_mape != float("inf") else None,
        }
        s3.put_object(
            Bucket=BUCKET,
            Key=manifest_key,
            Body=json.dumps(manifest, indent=2).encode(),
            ContentType="application/json",
        )
        print(f"✅ Nuevo campeón desplegado — MAPE: {mape:.2f}% (antes: {prev_mape:.2f}%)")
        print(f"   Manifest: s3://{BUCKET}/{manifest_key}")
    else:
        print(f"⏭️  Modelo NO desplegado — MAPE {mape:.2f}% no mejora al campeón {prev_mape:.2f}%")
        print(f"   El pickle fue guardado en S3 pero el manifest NO fue actualizado.")
        print(f"   Campeón activo sigue siendo: {prev.get('champion_model_key')}")

else:
    print("⚠️ R² <= 0 — modelo no guardado.")